# DAY 2 — 바이브 코딩 기반 현장 데이터 분석

**AI 적용을 위한 현장 데이터 수집 및 디지털화** · DAY 2 실습

교재 PART 02 (pp.62–126) 의 실습 코드를 코랩에서 바로 돌릴 수 있게 옮긴 것입니다.
설명과 그림은 교재와 학습사이트에 있습니다 — https://build-data.jobability.co.kr

---
### 코랩 쓰는 법 세 가지만

1. **셀 실행** — 코드 칸 왼쪽 ▶ 를 누르거나 `Shift + Enter`.
2. **위에서부터 차례로** — 앞 칸에서 만든 것을 뒤 칸이 씁니다. 건너뛰면 오류가 납니다.
3. **내 사본으로** — 「파일 → 드라이브에 사본 저장」을 먼저 해야 고친 내용이 남습니다.

> **제목의 번호는 교재 절 번호입니다.** 이론만 있는 절은 실행할 것이 없어 빠져 있어서
> 번호가 건너뜁니다(0 → 4 처럼). 빠진 절의 설명은 교재와 학습사이트에 있습니다.

> **API 키는 왼쪽 🔑(보안 비밀)에 한 번 등록해 두면 편합니다.** 이름을 `OPENAI_API_KEY` 로
> 적고 「노트북 액세스」를 켜면, 아래 키 칸이 묻지 않고 그냥 지나갑니다.

## 0. 실습 준비

이 아래 두 칸은 세션을 새로 열 때마다 한 번씩 실행합니다.

In [ ]:
# 그래프에 한글이 네모로 나오지 않게 폰트를 깝니다. 세션마다 한 번만 하면 됩니다.
!apt-get -qq install fonts-nanum > /dev/null 2>&1
import matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager
import glob, os
cand = glob.glob("/usr/share/fonts/truetype/nanum/NanumGothic*.ttf") + glob.glob("fonts/NanumGothic*.ttf")
if not cand:                      # apt 가 막힌 환경이면 자료 저장소에서 받아 씁니다
    import urllib.request
    os.makedirs("fonts", exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/aebonlee/materials/main/build-data/data/NanumGothic-Regular.ttf",
        "fonts/NanumGothic-Regular.ttf")
    cand = glob.glob("fonts/NanumGothic*.ttf")
if cand:
    font_manager.fontManager.addfont(cand[0])
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=cand[0]).get_name()
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트:", plt.rcParams["font.family"])

In [ ]:
# 실습 데이터 준비
# 1) 왼쪽 폴더 아이콘에 data_day2.zip 을 올려 두었으면 그것을 풉니다.
# 2) 없으면 같은 구조의 대체 데이터를 그 자리에서 만들어 씁니다.
import os, zipfile
if not os.path.exists("records"):
    if os.path.exists("data_day2.zip"):
        zipfile.ZipFile("data_day2.zip").extractall(".")
    else:
        print("data_day2.zip 이 없어 대체 실습 데이터를 만듭니다. 30초쯤 걸립니다.")
        import sys, subprocess, urllib.request
        urllib.request.urlretrieve("https://raw.githubusercontent.com/aebonlee/materials/main/build-data/data/make_day2_data.py", "make_day2_data.py")
        subprocess.run([sys.executable, "make_day2_data.py"], check=True)
print("준비된 폴더:", sorted(d for d in os.listdir(".") if os.path.isdir(d) and not d.startswith(".")))

## 1. 바이브 코딩의 원리

### 자연어 기반 코드 생성의 원리

**[2-1] 같은 작업을 지시하는 두 가지 방법**

```
[막연한 지시]
정비 데이터 좀 정리해 줘.
[구체적인 지시]
정비이력 CSV를 읽어서 행수, 컬럼, 앞 5행을 보여 줘.
각 컬럼의 자료형과 결측 개수도 정리해 줘.
```

## 4. 실습: 정비 이력 구조 확인

### 실습 준비: Colab 데이터 업로드

**[2-3] 데이터 압축 풀기**

In [ ]:
import os, zipfile
if not os.path.exists("records"):          # 아직 압축을 풀지 않았다면
    if os.path.exists("data_day2.zip"):
        zipfile.ZipFile("data_day2.zip").extractall(".")
    else:
        raise SystemExit("데이터가 없습니다 — 위쪽 「실습 데이터 준비」 칸을 먼저 실행하세요.")
print("데이터 준비 상태:", sorted(d for d in os.listdir(".") if os.path.isdir(d) and not
d.startswith(".")))

**[2-5] 분석 환경과 한글 폰트 설정**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
# 그래프 한글 폰트(나눔고딕, 동봉 파일)
import glob
_font = (glob.glob("fonts/NanumGothic*.ttf")
         + glob.glob("/usr/share/fonts/truetype/nanum/NanumGothic*.ttf"))
if _font:
    font_manager.fontManager.addfont(_font[0])
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=_font[0]).get_name()
else:
    print("한글 폰트를 찾지 못했습니다 — 맨 위 「한글 폰트」 칸을 먼저 실행하세요.")
plt.rcParams["axes.unicode_minus"] = False
print("pandas", pd.__version__, "| 폰트 설정 완료")

### 실습 시범: 데이터 기본 구조 확인

**[2-8] 정비 이력 파일 읽기**

In [ ]:
df = pd.read_csv("records/maintenance_2024_2025.csv")
print(df.shape)
df.head()

**[2-10] 컬럼 자료형 확인**

In [ ]:
df.info()

**[2-12] 컬럼별 결측 세기**

In [ ]:
df.isna().sum()

**[2-14] 정비 구분별 건수**

In [ ]:
df["구분"].value_counts()

**[2-16] 장비별 정비 건수**

In [ ]:
print("장비 대수:", df["장비ID"].nunique())
df["장비ID"].value_counts().head()

### 표기 불일치 컬럼 탐지

**[2-18] 정비기사 표기 세기**

In [ ]:
df["정비기사"].value_counts()

## 5. 실습: 데이터 전처리

### 표기 통일: 인명 변형 표기 정규화

**[2-21] 정비기사 표기 통일**

In [ ]:
mechanic_map = {"김반장": "김 반장", "박 기사님": "박 기사", "최주임": "최 주임"}
df["정비기사"] = df["정비기사"].replace(mechanic_map)
df["정비기사"].value_counts()

### 완전 중복 행의 탐지와 제거

**[2-23] 완전 중복 행 찾기**

In [ ]:
print("완전 중복 행:", df.duplicated().sum(), "건")
df[df.duplicated(keep=False)].sort_values("record_id").head(8)

**[2-25] 완전 중복 제거**

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print("제거 후 행수:", len(df))

### 유사 중복: 기계 후보 추출과 사람 판정

**[2-27] 짧은 간격으로 연달아 기록된 행 찾기**

In [ ]:
df["일시"] = pd.to_datetime(df["고장일시"])
s = df.sort_values(["장비ID", "일시"]).reset_index(drop=True)
gap = s.groupby("장비ID")["일시"].diff()          # 같은 장비의 직전 기록과의 간격
suspect = s[gap <= pd.Timedelta(minutes=30)]
print("유사 중복 의심:", len(suspect), "건")
suspect[["record_id", "장비ID", "고장일시", "증상", "정비시간(분)", "정비기사"]]

**[2-29] 의심 행과 직전 행 나란히 보기**

In [ ]:
# 의심 행과 그 직전 행을 나란히 놓고 눈으로 비교합니다.
pairs = sorted(set(suspect.index) | set(suspect.index - 1))
s.loc[pairs, ["record_id", "장비ID", "고장일시", "증상", "정비시간(분)", "정비기사"]]

### 이상치 검토 원칙

**[2-32] 정비시간 기초 통계**

In [ ]:
df["정비시간(분)"].describe()

**[2-34] 정비시간 박스플롯**

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.4))
try:
    ax.boxplot(df["정비시간(분)"].dropna(), orientation="horizontal")
except TypeError:                       # 옛 matplotlib
    ax.boxplot(df["정비시간(분)"].dropna(), vert=False)
ax.set_xlabel("정비시간(분)")
ax.set_title("정비시간 분포 — 오른쪽 끝의 점이 이상치 후보")
plt.tight_layout(); plt.show()

**[2-35] 정비시간 상위 기록 확인**

In [ ]:
df.sort_values("정비시간(분)", ascending=False)[
    ["record_id", "장비ID", "증상", "조치방법", "정비시간(분)"]].head(6)

### 결측 복원: 보조 데이터 기반 대체

**[2-38] 가동 기록 파일 읽기**

In [ ]:
kpi = pd.read_csv("kpi/operation_2024_2025.csv")
print(kpi.shape)
kpi.head(3)

**[2-40] 같은 날 다운타임으로 결측 채우기**

In [ ]:
df["일자"] = df["일시"].dt.strftime("%Y-%m-%d")
day_cnt = df.groupby(["장비ID", "일자"])["record_id"].transform("count")
down = kpi.set_index(["장비ID", "일자"])["다운타임(분)"]
key = list(zip(df["장비ID"], df["일자"]))
miss = df["정비시간(분)"].isna()
single = miss & (day_cnt == 1)                     # 그날 정비가 1건뿐인 결측
df.loc[single, "정비시간(분)"] = [down[k] for k in df.index[single].map(lambda i: key[i])]
print(f"결측 {miss.sum()}건 중 {single.sum()}건 복원, 남은 결측 {df['정비시간(분)'].isna().sum()}건")

## 6. 실습: 통계적 유의성 검정

### 계절성 검증: 카이제곱 검정

**[2-43] 겨울 ·저온성 증상 교차표**

In [ ]:
from scipy import stats
bd = df[df["구분"] == "고장수리"].copy()
bd["월"] = bd["일시"].dt.month
bd["겨울"] = bd["월"].isin([11, 12, 1, 2])
winter_kw = "시동|예열|동작 지연|반응 둔|수분|방전"
bd["저온성증상"] = bd["증상"].str.contains(winter_kw)
tbl = pd.crosstab(bd["겨울"], bd["저온성증상"])
tbl

**[2-45] 카이제곱 검정**

In [ ]:
chi2, p, dof, _ = stats.chi2_contingency(tbl)
print(f"카이제곱 = {chi2:.1f}, p = {p:.3e}")

### 기종별 정비시간 차이 검증: ANOVA와 비모수 검정

**[2-48] 장비ID로 기종 매핑하고 기종별 요약**

In [ ]:
eq_model = {
    "EX-101": "밥캣 E32",       "EX-102": "캐터필러 304 CR", "EX-103": "구보타 U27-4",
    "EX-104": "리파 R10-5",     "EX-201": "밥캣 E32",       "EX-202": "캐터필러 304 CR",
    "EX-203": "구보타 U27-4",   "EX-204": "밥캣 E32",       "EX-205": "리파 R10-5",
    "EX-206": "구보타 U27-4",   "EX-301": "캐터필러 304 CR", "EX-302": "밥캣 E32",
    "EX-303": "구보타 U27-4",   "EX-304": "리파 R10-5",     "EX-305": "밥캣 E32",
}
bd["기종"] = bd["장비ID"].map(eq_model)
bd.groupby("기종")["정비시간(분)"].agg(평균="mean", 중앙값="median", 건수="count").round(1)

**[2-50] ANOVA와 Kruskal-Wallis 검정**

In [ ]:
groups = [g.dropna().values for _, g in bd.groupby("기종")["정비시간(분)"]]
f, p_a = stats.f_oneway(*groups)
h, p_k = stats.kruskal(*groups)
print(f"ANOVA          F = {f:.2f}, p = {p_a:.3e}")
print(f"Kruskal-Wallis H = {h:.2f}, p = {p_k:.3e}")

### 특정 기종 대상 단측 검정

**[2-52] 리파 R10-5 단측 Mann-Whitney 검정**

In [ ]:
r10 = bd.loc[bd["기종"] == "리파 R10-5", "정비시간(분)"].dropna()
rest = bd.loc[bd["기종"] != "리파 R10-5", "정비시간(분)"].dropna()
u, p_mw = stats.mannwhitneyu(r10, rest, alternative="greater")
print(f"리파 중앙값 {r10.median():.0f}분 vs 나머지 {rest.median():.0f}분,  단측 p = {p_mw:.3e}")

### 부트스트랩(Bootstrap) 신뢰구간 추정

**[2-55] 부트스트랩 신뢰구간**

In [ ]:
rng = np.random.default_rng(42)
boot = [rng.choice(r10, size=len(r10), replace=True).mean() for _ in range(10_000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"리파 R10-5 평균 정비시간 {r10.mean():.0f}분 (95% CI {lo:.0f} ~ {hi:.0f}분, n={len(r10)})")

## 7. 실습: 가동률·MTBF 산출

### 전체 가동률과 월별 추이

**[2-59] 전체 가동률 계산**

In [ ]:
rate = kpi["실제가동시간"].sum() / kpi["계획가동시간"].sum() * 100
print(f"전체 가동률: {rate:.1f}%")

**[2-61] 월별 가동률 추이 그래프**

In [ ]:
kpi["월"] = kpi["일자"].str[:7]
monthly = (kpi.groupby("월")[["실제가동시간", "계획가동시간"]].sum()
             .eval("실제가동시간 / 계획가동시간 * 100"))
fig, ax = plt.subplots(figsize=(9, 3))
monthly.plot(ax=ax, marker="o")
ax.set_ylabel("가동률(%)"); ax.set_xlabel("")
ax.set_title("월별 가동률 추이 (2024-01 ~ 2025-11)")
plt.tight_layout(); plt.show()

### 기종별 가동률과 MTBF의 교차 해석

**[2-62] 기종별 가동률과 MTBF 계산**

In [ ]:
kpi["기종"] = kpi["장비ID"].map(eq_model)
by_model = kpi.groupby("기종").agg(
    실제=("실제가동시간", "sum"), 계획=("계획가동시간", "sum"),
    다운타임분=("다운타임(분)", "sum"), 고장=("고장건수", "sum"))
by_model["가동률(%)"] = (by_model["실제"] / by_model["계획"] * 100).round(1)
by_model["MTBF(h)"] = (by_model["실제"] / by_model["고장"]).round(1)
by_model["다운타임(h)"] = (by_model["다운타임분"] / 60).round(1)
by_model[["가동률(%)", "MTBF(h)", "다운타임(h)", "고장"]]

**[2-64] 기종별 가동률 ·MTBF 막대그래프**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
by_model["가동률(%)"].plot.bar(ax=axes[0], title="가동률(%)", rot=15)
by_model["MTBF(h)"].plot.bar(ax=axes[1], title="MTBF(시간)", rot=15)
plt.tight_layout(); plt.show()

### 이상 레코드 검토: 다운타임 초과 사례

**[2-66] 다운타임이 계획가동시간을 넘는 행 찾기**

In [ ]:
over = kpi[kpi["다운타임(분)"] / 60 > kpi["계획가동시간"]]
print(len(over), "건")
over[["일자", "장비ID", "계획가동시간", "실제가동시간", "다운타임(분)", "비고"]].head(8)

**[2-68] 초과 행의 비고 집계**

In [ ]:
over["비고"].value_counts()

**[2-70] 관리자 보고 요약 생성 지시**

```
아래 KPI 산출 결과를 정비팀장에게 보고할 한 장 요약으로 정리해 주세요.
[산출 결과]
- 전체 가동률: (오늘 구한 값)
- 기종별 MTBF: (기종별 값)
- 확인한 예외: (건수와 확인 결과)
[요구 사항]
- 결론 한 줄로 시작할 것
- 근거로 드는 지표는 세 개 이내로 고를 것
- 확인이 필요한 항목과 다음 조치를 마지막에 적을 것
- 산출 결과에 없는 수치는 쓰지 말 것
- 원인을 단정하지 말고 확인이 필요한 사항으로 적을 것
```

## 8. 실습: 부품 이력 파일 통합

### 4개 파일의 컬럼 구조 비교

**[2-72] 부품 파일 4종 읽고 컬럼 비교**

In [ ]:
hq  = pd.read_excel("parts/parts_usage_hq.xlsx")
bs  = pd.read_excel("parts/parts_usage_busan.xlsx")
gj  = pd.read_csv("parts/parts_issue_gwangju.csv")
dl  = pd.read_csv("parts/parts_usage_dealer.csv")
for name, d in [("본사", hq), ("부산", bs), ("광주", gj), ("딜러", dl)]:
    print(f"{name:3s} {len(d):5d}행  {list(d.columns)}")

**[2-74] 부산 파일 앞 3행**

In [ ]:
bs.head(3)

**[2-76] 광주 파일 앞 3행**

In [ ]:
gj.head(3)

### 주의 사례 1: 파일 소속과 장비 소속의 불일치

**[2-78] 본사 파일의 장비 소속 분포**

In [ ]:
site = {"1": "본사", "2": "부산지점", "3": "광주지점"}
hq["장비ID"].str[3].map(site).value_counts()

### 표준 스키마 변환 절차

**[2-80] 본사 파일 표준 스키마 변환**

In [ ]:
STD = ["사용일자", "장비ID", "부품번호", "부품명", "수량", "교체사유", "출처"]
h = hq.rename(columns={"수량(EA)": "수량"}).copy()
h["사용일자"] = pd.to_datetime(h["사용일자"])
h["출처"] = "본사"
h = h[STD]
h.head(2)

**[2-82] 부산 파일 표준 스키마 변환**

In [ ]:
b = bs.rename(columns={"일자": "사용일자", "장비": "장비ID", "품번": "부품번호",
                        "품명": "부품명", "개수": "수량", "사유": "교체사유"}).copy()
b["사용일자"] = pd.to_datetime(b["사용일자"], format="%Y.%m.%d")
b["수량"] = b["수량"].astype(str).str.replace("개", "").astype(int)   # '3개' → 3
b["출처"] = "부산지점"
b = b[STD]
b.head(2)

**[2-84] 광주 파일 표준 스키마 변환**

In [ ]:
master = pd.read_csv("parts/parts_master.csv")
g = gj.rename(columns={"date": "사용일자", "eq": "장비ID", "pn": "부품번호", "qty": "수량"}).copy()
g["사용일자"] = pd.to_datetime(g["사용일자"], format="%m/%d/%Y")
g = g.merge(master[["부품번호", "부품명"]], on="부품번호", how="left")   # 품번 → 부품명 복원
memo_map = {"정기교체": "정기교체", "정기 교체": "정기교체", "주기 도래": "정기교체",
            "고장": "고장수리", "고장수리": "고장수리", "수리건": "고장수리",
            "마모로 교체": "마모교체", "마모교체": "마모교체", "마모": "마모교체",
            "출고": "소모품 보충", "보충": "소모품 보충", "소모품": "소모품 보충"}
g["교체사유"] = g["memo"].map(memo_map)
g["출처"] = "광주지점"
g = g[STD]
g.head(2)

### 파일 통합 및 건수 검산

**[2-86] 세 파일 통합과 검산**

In [ ]:
parts = pd.concat([h, b, g], ignore_index=True)
print("통합:", len(parts), "행 (본사 2,000 + 부산 1,300 + 광주 1,000)")
print("결측:", parts.isna().sum().sum(), "칸 | 마스터에 없는 품번:",
      (~parts["부품번호"].isin(master["부품번호"])).sum(), "건")
parts["교체사유"].value_counts()

### 주의 사례 2: 딜러 파일 제외 판단

**[2-88] 딜러 파일 앞 3행**

In [ ]:
dl.head(3)

**[2-90] 통합본 저장**

In [ ]:
parts.to_csv("parts_usage_merged.csv", index=False, encoding="utf-8-sig")
print("저장 완료: parts_usage_merged.csv")

## 9. 실습: 고장 유형과 대응 우선순위

### 세 가지 기준의 상충

**단계 8. 파레토로 그려 보기**

---

## 마무리 — 오늘 코드를 파일로 받아 가기

아래 칸을 실행하면 **오늘 내가 실행한 코드 전부**가 `.py` 파일 하나로 내려받아집니다.
내가 고쳐 쓴 내용도 그대로 담기니, 회사에 돌아가 그대로 다시 돌려 볼 수 있습니다.

노트북 자체를 남기려면 「파일 → 드라이브에 사본 저장」도 함께 해 두세요.
막히는 곳은 학습사이트의 같은 절을 함께 보면 설명이 있습니다.

In [ ]:
# 오늘 실행한 코드를 파이썬 파일(.py) 한 장으로 내려받습니다.
# 이 칸은 실습을 다 끝낸 뒤 마지막에 한 번만 실행하세요.
import datetime
NL = chr(10)

runs = []
for n, cell in enumerate(In[1:], start=1):          # In = 지금까지 실행한 칸들
    if "files.download" in cell:                    # 이 칸 자신은 뺍니다
        continue
    keep = [ln for ln in cell.splitlines()          # 코랩 전용 명령(!apt-get 등)은 줄 단위로 뺍니다
            if not ln.lstrip().startswith(("!", "%"))]
    t = NL.join(keep).strip()
    if not t:                                       # 빈 칸도 뺍니다
        continue
    runs.append("# ─────────── 실행 " + str(n) + " ───────────" + NL + t + NL)

path = "DAY2_실습_내코드.py"
head = ("# DAY 2 실습 — 내가 실행한 코드 모음" + NL
        + f"# 저장 시각 {datetime.datetime.now():%Y-%m-%d %H:%M}" + NL
        + "# 코랩에서 실행한 순서 그대로입니다. 내가 고친 내용도 그대로 담깁니다." + NL + NL)
with open(path, "w", encoding="utf-8") as f:
    f.write(head + NL.join(runs))
print(path + " 저장 — 칸 " + str(len(runs)) + "개")

try:
    from google.colab import files
    files.download(path)                            # 내 PC로 내려받기
except ImportError:
    print("코랩이 아니면 왼쪽 폴더 아이콘에서 직접 내려받으세요.")